In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [2]:
# 데이터 로드
df = pd.read_csv("train.csv")

# 컬럼 삭제
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# NaN이 있는 행 삭제
df = df.dropna(axis=0)

# 레이블 인코더 객체 생성
le = LabelEncoder()

# 레이블 인코딩
sex = le.fit_transform(df['Sex'])
embarked = le.fit_transform(df['Embarked'])

# 기존 컬럼 삭제
df = df.drop(['Sex', 'Embarked'],axis=1)

# 새로운 컬럼에 인코딩한 결과 저장
df['Sex'] = sex
df['Embarked'] = embarked

df.info()

<class 'pandas.DataFrame'>
Index: 712 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  712 non-null    int64  
 1   Pclass    712 non-null    int64  
 2   Age       712 non-null    float64
 3   SibSp     712 non-null    int64  
 4   Parch     712 non-null    int64  
 5   Fare      712 non-null    float64
 6   Sex       712 non-null    int32  
 7   Embarked  712 non-null    int32  
dtypes: float64(2), int32(2), int64(4)
memory usage: 44.5 KB


In [3]:
from patsy import dmatrices
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor # VIF 계산 함수

def get_vif(formula, df):
    y, X = dmatrices(formula, df, return_type="dataframe")
    vif = pd.DataFrame()
    vif["VIF Factor"] = [variance_inflation_factor(X.values,i) for i in range(X.shape[1])]
    vif["features"] = X.columns
    return vif

In [4]:
formula = "Survived~" + "+".join(df.iloc[:,1:].columns) + "-1"
print(formula)
get_vif(formula, df)

Survived~Pclass+Age+SibSp+Parch+Fare+Sex+Embarked-1


,VIF Factor,features
0,5.990097,Pclass
1,4.169982,Age
2,1.638657,SibSp
3,1.618635,Parch
4,1.659422,Fare
5,3.035750,Sex
6,5.282948,Embarked


In [5]:
import statsmodels.formula.api as smf

model_titanic = smf.ols(formula=formula, data=df).fit()
model_titanic.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:               Survived   R-squared (uncentered):                   0.493
Model:                            OLS   Adj. R-squared (uncentered):              0.488
Method:                 Least Squares   F-statistic:                              97.81
Date:                Tue, 28 Apr 2026   Prob (F-statistic):                    1.58e-99
Time:                        17:40:01   Log-Likelihood:                         -446.48
No. Observations:                 712   AIC:                                      907.0
Df Residuals:                     705   BIC:                                      938.9
Df Model:                           7                                                  
Covariance Type:            nonrobust                                                  
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Pclass         0.1069      0.017      6.121      0.000       0.073       0.141
Age            0.0056      0.001      5.342      0.000       0.004       0.008
SibSp         -0.0132      0.021     -0.644      0.520      -0.054       0.027
Parch         -0.0023      0.023     -0.100      0.920      -0.047       0.042
Fare           0.0034      0.000      9.856      0.000       0.003       0.004
Sex           -0.4207      0.037    -11.288      0.000      -0.494      -0.347
Embarked       0.0698      0.022      3.161      0.002       0.026       0.113
==============================================================================
Omnibus:                       53.614   Durbin-Watson:                   1.855
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               35.947
Skew:                           0.431   Prob(JB):                     1.56e-08
Kurtosis:                       2.316   Cond. No.                         149.
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [6]:
formula = "Survived~Age+SibSp+Parch+Fare+Sex+Embarked-1"
print(formula)
get_vif(formula, df)

Survived~Age+SibSp+Parch+Fare+Sex+Embarked-1


,VIF Factor,features
0,3.928776,Age
1,1.583900,SibSp
2,1.560488,Parch
3,1.593394,Fare
4,2.680324,Sex
5,3.818932,Embarked


In [7]:
import statsmodels.formula.api as smf

model_titanic = smf.ols(formula=formula, data=df).fit()
model_titanic.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:               Survived   R-squared (uncentered):                   0.466
Model:                            OLS   Adj. R-squared (uncentered):              0.461
Method:                 Least Squares   F-statistic:                              102.6
Date:                Tue, 28 Apr 2026   Prob (F-statistic):                    1.10e-92
Time:                        17:40:01   Log-Likelihood:                         -464.91
No. Observations:                 712   AIC:                                      941.8
Df Residuals:                     706   BIC:                                      969.2
Df Model:                           6                                                  
Covariance Type:            nonrobust                                                  
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Age            0.0072      0.001      6.846      0.000       0.005       0.009
SibSp          0.0098      0.021      0.471      0.638      -0.031       0.050
Parch          0.0240      0.023      1.053      0.293      -0.021       0.069
Fare           0.0030      0.000      8.593      0.000       0.002       0.004
Sex           -0.3426      0.036     -9.541      0.000      -0.413      -0.272
Embarked       0.1410      0.019      7.321      0.000       0.103       0.179
==============================================================================
Omnibus:                       53.441   Durbin-Watson:                   1.829
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               36.939
Skew:                           0.444   Prob(JB):                     9.52e-09
Kurtosis:                       2.324   Cond. No.                         140.
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [8]:
formula = "Survived~Pclass+Age+SibSp+Parch+Fare+Sex+Embarked-1"
print(formula)
get_vif(formula, df)

Survived~Pclass+Age+SibSp+Parch+Fare+Sex+Embarked-1


,VIF Factor,features
0,5.990097,Pclass
1,4.169982,Age
2,1.638657,SibSp
3,1.618635,Parch
4,1.659422,Fare
5,3.035750,Sex
6,5.282948,Embarked


In [9]:
import statsmodels.formula.api as smf

model_titanic = smf.ols(formula=formula, data=df).fit()
model_titanic.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:               Survived   R-squared (uncentered):                   0.493
Model:                            OLS   Adj. R-squared (uncentered):              0.488
Method:                 Least Squares   F-statistic:                              97.81
Date:                Tue, 28 Apr 2026   Prob (F-statistic):                    1.58e-99
Time:                        17:40:01   Log-Likelihood:                         -446.48
No. Observations:                 712   AIC:                                      907.0
Df Residuals:                     705   BIC:                                      938.9
Df Model:                           7                                                  
Covariance Type:            nonrobust                                                  
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Pclass         0.1069      0.017      6.121      0.000       0.073       0.141
Age            0.0056      0.001      5.342      0.000       0.004       0.008
SibSp         -0.0132      0.021     -0.644      0.520      -0.054       0.027
Parch         -0.0023      0.023     -0.100      0.920      -0.047       0.042
Fare           0.0034      0.000      9.856      0.000       0.003       0.004
Sex           -0.4207      0.037    -11.288      0.000      -0.494      -0.347
Embarked       0.0698      0.022      3.161      0.002       0.026       0.113
==============================================================================
Omnibus:                       53.614   Durbin-Watson:                   1.855
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               35.947
Skew:                           0.431   Prob(JB):                     1.56e-08
Kurtosis:                       2.316   Cond. No.                         149.
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [10]:
import numpy as np

df2 = df.copy()
df2["Fare"] = np.log1p(df2["Fare"])
df2

In [13]:
formula = "Survived~" + "+".join(df2.iloc[:,1:].columns) + "-1"
print(formula)
get_vif(formula, df2)

Survived~Pclass+Age+SibSp+Parch+Fare+Sex+Embarked-1


,VIF Factor,features
0,5.821890,Pclass
1,5.941821,Age
2,1.806010,SibSp
3,1.669711,Parch
4,7.574309,Fare
5,3.042677,Sex
6,5.359209,Embarked


In [14]:
import statsmodels.formula.api as smf

model_titanic = smf.ols(formula=formula, data=df2).fit()
model_titanic.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:               Survived   R-squared (uncentered):                   0.605
Model:                            OLS   Adj. R-squared (uncentered):              0.601
Method:                 Least Squares   F-statistic:                              154.4
Date:                Tue, 28 Apr 2026   Prob (F-statistic):                   1.12e-137
Time:                        17:40:01   Log-Likelihood:                         -357.24
No. Observations:                 712   AIC:                                      728.5
Df Residuals:                     705   BIC:                                      760.5
Df Model:                           7                                                  
Covariance Type:            nonrobust                                                  
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Pclass         0.0425      0.015      2.796      0.005       0.013       0.072
Age           -0.0030      0.001     -2.702      0.007      -0.005      -0.001
SibSp         -0.0956      0.019     -5.025      0.000      -0.133      -0.058
Parch         -0.0565      0.020     -2.780      0.006      -0.096      -0.017
Fare           0.2348      0.013     18.046      0.000       0.209       0.260
Sex           -0.4429      0.033    -13.457      0.000      -0.508      -0.378
Embarked       0.0181      0.020      0.921      0.358      -0.020       0.057
==============================================================================
Omnibus:                       27.287   Durbin-Watson:                   1.866
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               28.680
Skew:                           0.470   Prob(JB):                     5.92e-07
Kurtosis:                       2.710   Cond. No.                         74.7
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""